Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from scipy.ndimage import histogram
import re
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import pickle

In [ ]:
data = pd.read_csv('bbc-text.csv')

In [ ]:
data.head()

Preproccesing data(lowercasing,deleting numbers,stop words and other symbols, stemming)

In [ ]:
data.dropna()

In [ ]:
def lowercase_text(text):
    if isinstance(text, str):
        return text.lower()
    return text

In [ ]:
def remove_numbers(text):
    if isinstance(text, str):
        return re.sub(r'\d+', '', text)
    return text

In [ ]:
symbols_pattern = r"[.,!?;:\"'\(\)\[\]\{\}@#$%^&*_\+=/\\|~]"
def remove_symbols(text):
    if isinstance(text, str):
        return re.sub(symbols_pattern, '', text)
    return text

In [ ]:
def remove_extra_spaces(text):
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text)
    return text

In [ ]:
data['text'] = data['text'].apply(lowercase_text)
data['text'] = data['text'].apply(remove_numbers)
data['text'] = data['text'].apply(remove_symbols)
data['text'] = data['text'].apply(remove_extra_spaces)

In [ ]:
data.sample(n=10)

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return filtered_tokens
data['text'] = data['text'].apply(remove_stopwords)

In [ ]:
stemmer = PorterStemmer()
def stemming(tokens_list):
    return [stemmer.stem(token) for token in tokens_list]

data['text'] = data['text'].apply(stemming)

In [ ]:
data['text'] = data['text'].apply(lambda tokens: ' '.join(tokens))

Vectorization

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, max_df=0.95)
x = vectorizer.fit_transform(data['text'])

encoder = LabelEncoder()
y = encoder.fit_transform(data['category'])

In [ ]:
x.shape

In [ ]:
print(vectorizer.get_feature_names_out())
x.toarray()

Train/test split


In [ ]:
train_x, test_x,train_y, test_y = train_test_split(x, y, test_size=0.2, random_state=1, shuffle = True)


In [ ]:
print('train_x : ')
print(train_x)
print('')
print('test_x : ')
print(test_x)
print('')
print('train_y : ')
print(train_y)
print('')
print('test_y : ')
print(test_y)

Creating model(For the first i will use Multinomial Naive Bayes)

In [ ]:
model = MultinomialNB()
model.fit(train_x, train_y)

Predicting

In [ ]:
pred_y = model.predict(test_x)
accuracy = accuracy_score(test_y, pred_y)
print(f'Accuracy: {accuracy*100:.2f}%\n')

In [ ]:
"""""
with open("nb_model.pkl", "wb") as f:
    pickle.dump(model, f)
"""""

Alternative model (Logistic Regression)

In [ ]:
alternative_model = LogisticRegression()
alternative_model.fit(train_x, train_y)

In [ ]:
alt_pred_y = alternative_model.predict(test_x)
alternative_model_accuracy = accuracy_score(test_y, alt_pred_y)
print(f'Accuracy: {alternative_model_accuracy*100:.2f}%\n')